In [ ]:
# Fig 2c/d (as of 2025-05-23)

## Initialization

### Imports

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
from scipy.optimize import curve_fit

from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd, load_parquet_signals
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.processing.figure_of_merit import gaussian
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.loading.spectrum_unfolding import load_neutron_response_matrix
from data_processing.helpers import get_midpoints_from_bins, stop
from data_processing.processing.spectrum_unfolding import NDHistogram, unfold_spectrum, _nan_divide

### Functions

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: NeutronStrategyFactory,
    window_type: WindowType,
    loading: bool,
    settings: NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data


In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

## Data Loading

### Loading Params

In [ ]:
background_id = "TB-46"
reactor_id = "TB-26"
experiment_ids = [background_id, reactor_id]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = helpers.get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = (
    ExperimentDataKey.NEW_CALIBRATION
    if is_new_calibration
    else ExperimentDataKey.CAEN_CALIBRATION
)

In [ ]:
# detector_code = helpers.get_input_required(
#     """\
# Which detector was used?
# 1: Original detector (detector 1)
# 2: New detector (detector 2)
# """,
#     [Detector.ZERO, Detector.ONE],
#     lambda x: Detector(int(x)-1)
# )
detector_code = Detector.ZERO

In [ ]:
default_fit_input = 2  # changed to peak finder mode, approved by Fatima 2024-07-18
# fit_input = helpers.get_input_with_default(
#     """\
# Which bimodal fit type do you want to use?
# 1: Bounds based
# 2: Peak finder based (default)
# Press Enter for default
# """,
#     default_fit_input,
#     int
# )
fit_input = 2

fit_styles: dict[int, SliceFitStyle] = {
    1: "bounds",
    2: "peak_finder"
}
fit_style = fit_styles.get(fit_input, fit_styles[default_fit_input])

In [ ]:
# kind of window (Nasa, N distribution)
# load or generate
# specific settings for each condition to make namedtuple
# - generator settings (i.e. sigma, etc.)
# - file path prefix for loading
# done = False
strategy_factory = NeutronStrategyFactory()

# while not done:
#     window_input = helpers.get_input_with_default(
#         """\
# Which neutron classification window do you want to use?
# 1: NASA window (default)
# 2: Neutron distribution window
# Press Enter for default
# """,
#         1,
#         int
#     )
#     load_window_input = helpers.get_input_with_default(
#         """\
# Do you want to load the borders from the standard border file?
# [y/n, or press Enter for no]
# """,
#         "n",
#         str
#     )
#     done = True
#     will_load = load_window_input == "y"

#     try:
#         if window_input == 1:
#             if will_load:
#                 settings = get_nasa_loading_settings(calib_key=calib_key)
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "nasa", True, settings
#                 )
#             else:
#                 settings = get_nasa_generation_settings(calib_key=calib_key)
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "nasa", False, settings
#                 )
#                 pass
#         elif window_input == 2:
#             if will_load:
#                 settings = get_n_distro_loading_settings(calib_key=calib_key)
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "n_distro", True, settings
#                 )
#             else:
#                 settings = get_n_distro_generation_settings()
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "n_distro", False, settings
#                 )
#         else:
#             print("Invalid classification window type given, please try again")
#             done = False
#     except ValueError as err:
#         print("Problem found:")
#         print(err)
#         print("Please try again")
#         done = False

settings = NasaGenerationSettings(
    window_offset=0.2,
    sigma=5,
    lower_energy_bound=0.05,
    recalculate_lower_energy_bound=False
)
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings
)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

### Loading

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load_parquet_psd(exp_id)
    exp_data["signals_df"] = load_parquet_signals(exp_id)

## Processing

### Initial Processing

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = recalibrate(unclassified_df, detector_code)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

### Neutron Classification

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

### Signal Processing

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    signals_df = exp_data["signals_df"]
    signals_df = signals_df.loc[psd_report.index].astype("int32")

    signals_np = signals_df.to_numpy()
    baselines = signals_np[:, :30].mean(axis=1).reshape(-1, 1)
    signals_np = -signals_np + baselines
    signals_df = pd.DataFrame(
        signals_np,
        index=signals_df.index,
        columns=signals_df.columns
    )
    psd_report["peak_height"] = signals_df.max(axis=1)
    print(psd_report["peak_height"].max())
    print(psd_report["peak_height"].min())
    
    exp_data["signals_df"] = signals_df
    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value

    gamma_only = psd_report.query(f"~{n_class_col_name}").copy()
    neutrons_only = psd_report.query(n_class_col_name).copy()

    # print(psd_report.shape)
    # print(neutrons_only.shape)
    # print(gamma_only.shape)
    
    exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only

### Pulse Energy Distribution

In [ ]:
bin_width = 50
for exp_id, exp_data in experiment_neutron_data.items():
    # print(exp_id)
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    n_energy = neutrons_only["ENERGY"]
    all_particles = exp_data[ExperimentDataKey.PSD_REPORT]
    all_energy = all_particles["ENERGY"]

    # print(n_energy.min(), n_energy.max())
    # print(all_energy.min(), all_energy.max())
    bins = np.arange(0, 4096, step=bin_width)
    Zn, *_ = np.histogram(n_energy, bins=bins)
    Z, *_ = np.histogram(all_energy, bins=bins)

    exp_data["pulse_energy_distro"] = {
        "neutron": {"counts": Zn, "bins": bins},
        "all": {"counts": Z, "bins": bins},
    }

In [ ]:
# # moving average
# for exp_id, exp_data in experiment_neutron_data.items():
#     phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
#     phd_nh_histogram = phd_histogram_data["neutron"]["counts"]
#     phd_subh_histogram = phd_histogram_data["all"]["counts"]
    
#     phd_nh_moving_average = moving_average_centered(phd_nh_histogram)
#     phd_subh_moving_average = moving_average_centered(phd_subh_histogram)
    
#     phd_histogram_data["neutron"]["moving_average"] = phd_nh_moving_average
#     phd_histogram_data["subset"]["moving_average"] = phd_subh_moving_average
#     exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = phd_histogram_data

In [ ]:
exp_data = experiment_neutron_data[background_id]
phd_histogram_data = exp_data["pulse_energy_distro"]["all"]
Z = phd_histogram_data["counts"]
bins = phd_histogram_data["bins"]

bin_mids = (bins[1:] + bins[:-1]) / 2

bin_above_lo = bin_mids >= 2300
bin_below_hi = bin_mids <= 3500
bin_mask = bin_above_lo & bin_below_hi

masked_bins = bin_mids[bin_mask]
print(masked_bins)
masked_Z = Z[bin_mask]
print(masked_Z)

fit_params, *_ = curve_fit(gaussian, masked_bins, masked_Z, p0=(2400, 200, 30000))
print(fit_params)
phd_histogram_data["fit_params"] = fit_params

### PHDs

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    peak_height = psd_report["peak_height"]
    n_peak_height = neutrons_only["peak_height"]
    # print(neutron_energies.max())
    # print(neutron_energies.min())

    bin_start = 0
    bin_end = 16000
    bin_size = 50
    bins = np.arange(bin_start, bin_end+bin_size, step=bin_size)
    
    Z, *_ = np.histogram(peak_height, bins=bins)
    Zn, *_ = np.histogram(n_peak_height, bins=bins)
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = {
        "all": {"counts": Z, "bins": bins},
        "neutron": {"counts": Zn, "bins": bins},
    }

In [ ]:
# # moving average
# for exp_id, exp_data in experiment_neutron_data.items():
#     phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
#     phd_g_histogram = phd_histogram_data["all"]["counts"]
#     # phd_g_histogram = phd_histogram_data["gamma"]["standard"]

#     window_length = 17
#     phd_g_moving_average = moving_average_centered(phd_g_histogram, n=window_length)
#     # phd_g_moving_average = moving_average_centered(phd_g_histogram)

#     phd_histogram_data["all"]["moving_average"] = phd_g_moving_average
#     # phd_histogram_data["gamma"]["moving_average"] = phd_g_moving_average
#     exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = phd_histogram_data

In [ ]:
exp_data = experiment_neutron_data[background_id]
phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["all"]
Z = phd_histogram_data["counts"]
bins = phd_histogram_data["bins"]

bin_mids = (bins[1:] + bins[:-1]) / 2

bin_above_lo = bin_mids >= 11000
bin_below_hi = bin_mids <= 15500
bin_mask = bin_above_lo & bin_below_hi

masked_bins = bin_mids[bin_mask]
# print(masked_bins)
masked_Z = Z[bin_mask]
# print(masked_Z)

fit_params, *_ = curve_fit(gaussian, masked_bins, masked_Z, p0=(12000, 1000, 10000))
print(fit_params)
phd_histogram_data["fit_params"] = fit_params

## Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"
bg_green = "#21d894"

In [ ]:
annot_x_offset = 40
annot_y_offset = 1500
fig, ax = plt.subplots(figsize=(12, 8))

fit_params = None
for exp_id, exp_data in experiment_neutron_data.items():
    if exp_id != background_id:
        continue
    plot_key = "all" if exp_id == background_id else "neutron"
    color = bg_blue if exp_id == background_id else bg_red

    plot_data = exp_data["pulse_energy_distro"][plot_key]
    Z = plot_data["counts"]
    bins = plot_data["bins"]
    _fit_params = plot_data.get("fit_params")
    if _fit_params is not None:
        fit_params = _fit_params

    bin_mids = (bins[1:] + bins[:-1]) / 2

    ax.fill_between(bin_mids, Z, color=color, alpha=0.5)

# if fit_params is not None:
#     mu, sigma, _ = fit_params
#     gaussian_x = np.linspace(mu - 3 * sigma, mu + 3 * sigma, 300)
#     gaussian_y = gaussian(gaussian_x, *fit_params)
#     ax.plot(gaussian_x, gaussian_y, lw=2, color="black")

# ax.vlines([mu, mu+sigma], 0, 110000, lw=2, ls=":", color=bg_red)
# ax.annotate(
#     r"$\mu$",
#     (mu+annot_x_offset, 110000-annot_y_offset),
#     ha="left",
#     va="top",
#     fontsize=fontsize-2
# )
# ax.annotate(
#     r"$\mu+\sigma$",
#     (mu+sigma+annot_x_offset, 110000-annot_y_offset),
#     ha="left",
#     va="top",
#     fontsize=fontsize-2
# )

ax.set_ylim(0, 110000)
ax.set_xlabel("Pulse energy (ADC channel x 1000)", fontsize=fontsize)
ax.set_ylabel("Counts (x 1000)", fontsize=fontsize)
ax.xaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
ax.yaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
ax.tick_params(labelsize=fontsize)

In [ ]:
annot_x_offset = 100
annot_y_offset = 200
fig, ax = plt.subplots(figsize=(12, 8))

fit_params = None
for exp_id, exp_data in experiment_neutron_data.items():
    # if exp_id != background_id:
    #     continue
    plot_key = "all" if exp_id == background_id else "neutron"
    color = bg_blue if exp_id == background_id else bg_red

    plot_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION][plot_key]
    Z = plot_data["counts"]
    bins = plot_data["bins"]
    print(exp_id, bins.shape, Z.shape)
    _fit_params = plot_data.get("fit_params")
    if _fit_params is not None:
        fit_params = _fit_params

    bin_mids = (bins[1:] + bins[:-1]) / 2

    mu, sigma, _ = fit_params
    gaussian_x = np.linspace(mu - 3 * sigma, mu + 3 * sigma, 300)
    gaussian_y = gaussian(gaussian_x, *fit_params)

    ax.fill_between(bin_mids, Z, color=color, alpha=0.5)

# print(fit|

# ax.vlines([mu, mu+sigma], 0, 20000, lw=2, ls=":", color=bg_red)
# ax.annotate(
#     r"$\mu$",
#     (mu+annot_x_offset, 20000-annot_y_offset),
#     ha="left",
#     va="top",
#     fontsize=fontsize-2
# )
# ax.annotate(
#     r"$\mu+\sigma$",
#     (mu+sigma+annot_x_offset, 20000-annot_y_offset),
#     ha="left",
#     va="top",
#     fontsize=fontsize-2
# )

ax.set_ylim(0, 35000)
ax.set_xlim(0, 18000)
ax.set_xlabel("Pulse height (ADC channel x 1000)", fontsize=fontsize)
ax.set_ylabel("Counts (x 1000)", fontsize=fontsize)
ax.xaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
ax.yaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
ax.tick_params(labelsize=fontsize)

In [ ]:
input("Processing done, hit Enter to finish")
stop()